# E14S + E15S: Harmony-integrated UMAP and cell-type annotation

This notebook loads two Cell Ranger `filtered_feature_bc_matrix.h5` outputs (gene expression plus antibody capture / spatial barcodes), assigns each cell to a spatial barcode pair using the same row/column barcode layout as in `spatial_assigning.ipynb`, merges both samples into one `AnnData`, and builds a **single UMAP** after **Harmony** batch correction on PCA (`scanpy.external.pp.harmony_integrate`).

Demultiplexing utilities live in `spatial_barcode_demux.py` (`demux_all_cells`).

**Dependencies:** `scanpy`, `anndata`, `harmonypy`, `leidenalg`, `pandas`, `numpy`, `matplotlib`. If Harmony fails with a NumPy / PyTorch error, install the classic implementation: `pip install 'harmonypy==0.0.10'` (newer `harmonypy` may use PyTorch and clash with NumPy 2.x in some setups).

In [ ]:
%pip install -q 'harmonypy==0.0.10' leidenalg

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import scanpy.external as sce
import anndata as ad

from spatial_barcode_demux import demux_all_cells

warnings.filterwarnings("ignore", category=UserWarning)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

SEED = 42
np.random.seed(SEED)

## Paths and spatial barcode layout

Adjust `ROWS` / `COLS` if your chip layout differs (see commented options in `spatial_assigning.ipynb`). `ascend` controls ordering when mapping barcode names to integer coordinates for plotting.

In [ ]:
ROOT = Path("/ix1/ylee/shared/MC38_Hypoxia_001").resolve()

SAMPLES = {
    "E14S": ROOT / "E14S" / "filtered_feature_bc_matrix.h5",
    "E15S": ROOT / "E15S" / "filtered_feature_bc_matrix.h5",
}

ROWS = [f"sbc{i}" for i in range(1, 43)]
COLS = [f"sbc{i}" for i in range(97, 139)]
ASCEND = True
DEMUX_THRESHOLD_K = 1.5

for name, p in SAMPLES.items():
    assert p.is_file(), f"Missing matrix: {p}"

## Load each sample, tag barcodes, concatenate

10x barcodes can overlap across lanes; suffixing with the sample id keeps observations unique before `concat`.

In [ ]:
def load_tagged_matrix(path: Path, sample_id: str) -> ad.AnnData:
    adata = sc.read_10x_h5(path, gex_only=False)
    adata.var_names_make_unique()
    adata.obs_names = adata.obs_names.astype(str) + "_" + sample_id
    adata.obs["sample"] = sample_id
    return adata


adatas = [load_tagged_matrix(p, sid) for sid, p in SAMPLES.items()]
adata = ad.concat(adatas, join="outer", merge="same")
adata.obs_names_make_unique()
adata

## Spatial barcode demux → `spatial_x`, `spatial_y`, `spatial_label`

We take antibody-capture counts for the configured `ROWS` and `COLS`, run `demux_all_cells`, and record the winning barcode on each axis. Low-confidence cells are optionally re-run with the same threshold (mirroring the workflow in `spatial_assigning.ipynb`).

In [ ]:
def spatial_assign(adata: ad.AnnData) -> pd.DataFrame:
    prot = adata[:, adata.var["feature_types"] == "Antibody Capture"].copy()
    prot_df = prot.to_df()
    keep = [c for c in prot_df.columns if c in ROWS or c in COLS]
    prot_sub = prot_df[keep]

    xy = demux_all_cells(prot_sub, threshold_k=DEMUX_THRESHOLD_K).set_index("cell")
    xy = xy.rename(columns={"x": "spatial_x", "y": "spatial_y"})

    def label_row(r):
        if r["confidence_1"] >= 0.1 and r["confidence_2"] >= 0.1:
            return "confident"
        return "non-confident"

    xy["spatial_label"] = xy.apply(label_row, axis=1)

    non_conf = xy.index[xy["spatial_label"] == "non-confident"]
    conf = xy.index[xy["spatial_label"] != "non-confident"]
    if len(non_conf):
        redo = demux_all_cells(prot_sub.loc[non_conf], threshold_k=DEMUX_THRESHOLD_K)
        redo = redo.set_index("cell")[["x", "y"]].rename(columns={"x": "spatial_x", "y": "spatial_y"})
        redo["spatial_label"] = "non-confident"
        xy = pd.concat([xy.loc[conf], redo])

    return xy[["spatial_x", "spatial_y", "spatial_label"]]


xy_parts = []
for a in adatas:
    xy_parts.append(spatial_assign(a))
xy_all = pd.concat(xy_parts)

adata.obs["spatial_x"] = xy_all.reindex(adata.obs_names)["spatial_x"]
adata.obs["spatial_y"] = xy_all.reindex(adata.obs_names)["spatial_y"]
adata.obs["spatial_label"] = xy_all.reindex(adata.obs_names)["spatial_label"]

adata.obs["spatial_label"].value_counts()

## RNA preprocessing, PCA, Harmony, neighbors, UMAP, Leiden

Harmony adjusts the PCA embedding using the `sample` column; the neighbor graph and UMAP are computed on **`X_pca_harmony`** so the layout reflects batch-corrected structure.

In [ ]:
rna = adata[:, adata.var["feature_types"] == "Gene Expression"].copy()

sc.pp.filter_cells(rna, min_genes=200)
sc.pp.filter_genes(rna, min_cells=3)

sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)

sc.pp.highly_variable_genes(rna, n_top_genes=3000, batch_key="sample", subset=True)
rna.raw = rna
rna = rna[:, rna.var.highly_variable].copy()

sc.pp.scale(rna, max_value=10)
sc.tl.pca(rna, svd_solver="arpack")

sce.pp.harmony_integrate(rna, key="sample", basis="X_pca", adjusted_basis="X_pca_harmony")

sc.pp.neighbors(rna, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=40)
sc.tl.umap(rna)
sc.tl.leiden(rna, resolution=0.6, key_added="leiden")

rna

## Check batch mixing on UMAP

After Harmony, `sample` should not dominate clusters (qualitative check).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sc.pl.umap(rna, color="sample", ax=axes[0], show=False, title="UMAP (sample)")
sc.pl.umap(rna, color="leiden", ax=axes[1], show=False, title="UMAP (Leiden)", legend_loc="on data")
plt.tight_layout()
plt.show()

## Marker-based exploration and coarse annotation

Curated mouse panels below are a starting point for MC38 / tumor microenvironment work. Refine `LEIDEN_TO_CELLTYPE` using `sc.pl.dotplot` and `sc.tl.rank_genes_groups` outputs.

In [ ]:
markers = {
    "T_cell": ["Cd3e", "Cd3d", "Trac"],
    "NK": ["Nkg7", "Klrb1c", "Gzma"],
    "B_cell": ["Cd79a", "Ms4a1", "Cd19"],
    "Myeloid": ["Lyz2", "C1qa", "Csf1r"],
    "Neutrophil": ["S100a8", "S100a9", "Camp"],
    "DC": ["Zbtb46", "Flt3"],
    "Endothelial": ["Pecam1", "Cdh5", "Kdr"],
    "Fibroblast": ["Col1a1", "Dcn", "Pdgfra"],
    "Epithelial_tumor": ["Epcam", "Krt8", "Krt18"],
}

present = {k: [g for g in v if g in rna.raw.var_names] for k, v in markers.items()}
flat = [g for genes in present.values() for g in genes]

sc.pl.dotplot(rna, var_names=present, groupby="leiden", dendrogram=True, standard_scale="var")

sc.tl.rank_genes_groups(rna, groupby="leiden", method="wilcoxon", use_raw=True)
sc.pl.rank_genes_groups_matrixplot(rna, n_genes=5, groupby="leiden", use_raw=True)

result = sc.get.rank_genes_groups_df(rna, group=None)
result.groupby("group").head(8)

### Manual label map (edit after inspecting dot plots)

Replace the placeholder mapping with your own `leiden` → cell type assignments.

In [ ]:
LEIDEN_TO_CELLTYPE = {
    str(c): f"cluster_{c}" for c in sorted(rna.obs["leiden"].unique(), key=lambda x: int(str(x)))
}

rna.obs["celltype"] = rna.obs["leiden"].map(LEIDEN_TO_CELLTYPE).astype("category")

sc.pl.umap(rna, color=["celltype", "leiden", "sample"], ncols=3)

## Optional: spatial scatter of annotated cells

Integer grid positions follow the same idea as `spatial_assigning.ipynb`: order unique `spatial_x` / `spatial_y` barcode names, map to 1…N, add small jitter for duplicates.

In [ ]:
from collections import defaultdict

mask = rna.obs["spatial_x"].notna() & rna.obs["spatial_y"].notna()
sp = rna[mask].copy()

rows = sorted(sp.obs["spatial_x"].unique(), key=lambda s: int("".join(filter(str.isdigit, s))))
cols = sorted(sp.obs["spatial_y"].unique(), key=lambda s: int("".join(filter(str.isdigit, s))))
if not ASCEND:
    rows = rows[::-1]
    cols = cols[::-1]

row_map = {bc: i + 1 for i, bc in enumerate(rows)}
col_map = {bc: i + 1 for i, bc in enumerate(cols)}
cx = sp.obs["spatial_x"].map(row_map).to_numpy(dtype=float)
cy = sp.obs["spatial_y"].map(col_map).to_numpy(dtype=float)

jitter_radius = 0.3
groups = defaultdict(list)
for i, key in enumerate(zip(cx, cy)):
    groups[key].append(i)
for idxs in groups.values():
    n = len(idxs)
    if n <= 1:
        continue
    for rank, i in enumerate(idxs):
        ang = 2 * np.pi * rank / n
        cx[i] += jitter_radius * np.cos(ang)
        cy[i] += jitter_radius * np.sin(ang)

sp.obsm["X_spatial"] = np.column_stack([cx, cy])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.embedding(sp, basis="spatial", color="celltype", ax=axes[0], show=False, title="Spatial layout (celltype)")
sc.pl.umap(sp, color="celltype", ax=axes[1], show=False, title="UMAP (celltype)")
plt.tight_layout()
plt.show()

## Save

Writes an h5ad next to this notebook; edit the path if you prefer another location.

In [ ]:
out = ROOT / "combined_E14S_E15S_harmony_umap.h5ad"
rna.write_h5ad(out)
out